# 🌿 Pipeline de Cuidado de Planta — Simulador

Este notebook muestra el flujo completo de interacción con el backend:

1. Crear una planta nueva
2. Pipeline ideal: cuidados óptimos → salud sube, planta crece
3. Pipeline de descuido: riego excesivo → pudrición de raíces → salud baja → crecimiento se frena y **retrocede**
4. Recuperación: corregir condiciones → salud sube → crecimiento retoma
5. Gestión de sustrato, maceta e historial

> **Requisito:** el backend FastAPI debe estar corriendo en `http://localhost:8000`

In [1]:
import requests
import json
from pprint import pprint

BASE = "http://127.0.0.1:8000"

def mostrar(resp: dict, titulo: str = ""):
    """Imprime la respuesta del API de forma legible."""
    if titulo:
        print(f"\n{'═'*55}")
        print(f"  {titulo}")
        print(f"{'═'*55}")
    print(json.dumps(resp, indent=2, ensure_ascii=False, default=str))

print("Librerías listas ✓")

Librerías listas ✓


## 1. Autenticación

Primero obtenemos un token de acceso para poder crear la planta.

In [2]:

# ── Ajusta con tus credenciales ──────────────────────────────────────────────
CORREO   = "miss@hotmail.com"
PASSWORD = "12345678"
# ─────────────────────────────────────────────────────────────────────────────

resp = requests.post(
    f"{BASE}/auth/login",
    json={"correo": CORREO, "password": PASSWORD},
)

print(f"Status HTTP: {resp.status_code}")
data = resp.json()

if resp.status_code == 200:
    u       = data["usuario"]
    USER_ID = u["id"]
    TOKEN   = None          # este backend no usa JWT — la sesión es por estado online
    HEADERS = {}
    print(f"\n✓ Login exitoso")
    print(f"  ID      : {u['id']}")
    print(f"  Nombre  : {u['nombre']}")
    print(f"  Correo  : {u['correo']}")
    print(f"  Online  : {u['online']}")
else:
    print(f"\n✗ Error: {data.get('mensaje', data)}")
    USER_ID = None
    HEADERS = {}


Status HTTP: 200

✓ Login exitoso
  ID      : 16
  Nombre  : Miss
  Correo  : miss@hotmail.com
  Online  : True


## 2. Crear la planta

Si el usuario no tiene plantas aún, el backend la marca como `es_nuevo=True`.

In [4]:
crear_resp = requests.post(
    f"{BASE}/plants",
    json={"nombre": "Orquídea Dendrobium", "tipo": "orquidea", "user_id": USER_ID},
    headers=HEADERS,
).json()

mostrar(crear_resp, "Planta creada")

PLANT_ID = crear_resp.get("planta", {}).get("id_plant")
print(f"\nPlant ID: {PLANT_ID}")



═══════════════════════════════════════════════════════
  Planta creada
═══════════════════════════════════════════════════════
{
  "mensaje": "Planta 'Orquídea Dendrobium' creada exitosamente.",
  "planta": {
    "id_plant": 1,
    "plant_name": "Orquídea Dendrobium",
    "plant_type": "orquidea",
    "water_level": 30.0,
    "light_level": 40.0,
    "humidity_level": 60.0,
    "health": 76.67,
    "growth_stage": "germinacion",
    "total_care_actions": 0,
    "creation_date_plant": "2026-04-19",
    "fk_user_id": 16
  }
}

Plant ID: 1


## 3. Estado inicial

La planta nace con valores conservadores (agua=30, luz=40, humedad=60, ventilación=50).
La salud inicial no es perfecta — hay espacio para mejorar.

In [5]:
estado = requests.get(f"{BASE}/plants/{PLANT_ID}/status", headers=HEADERS).json()
mostrar(estado, "Estado inicial")

p = estado["planta"]
print(f"""
  Agua        : {p['water_level']}%
  Luz         : {p['light_level']}%
  Humedad     : {p['humidity_level']}%
  Ventilación : {p['ventilation_level']}%
  Salud       : {p['health']}  ({estado['salud_etiqueta']})
  Etapa       : {p['growth_stage']}
  Sustrato    : {p['substrate_name']}
""")


═══════════════════════════════════════════════════════
  Estado inicial
═══════════════════════════════════════════════════════
{
  "planta": {
    "id_plant": 1,
    "plant_name": "Orquídea Dendrobium",
    "plant_type": "orquidea",
    "water_level": 30.0,
    "light_level": 40.0,
    "humidity_level": 60.0,
    "ventilation_level": 50.0,
    "health": 76.67,
    "growth_stage": "germinacion",
    "total_care_actions": 0,
    "creation_date_plant": "2026-04-19",
    "substrate_name": "mixto"
  },
  "salud_etiqueta": "Buena",
  "etapa_descripcion": "La semilla ha germinado. La planta está iniciando su vida.",
  "alertas": [
    "💧 Agua baja (30.0%): Marchitamiento de hojas, raíces en estrés hídrico."
  ]
}

  Agua        : 30.0%
  Luz         : 40.0%
  Humedad     : 60.0%
  Ventilación : 50.0%
  Salud       : 76.67  (Buena)
  Etapa       : germinacion
  Sustrato    : mixto



---
## 4. 🟢 Pipeline ideal — Cuidados óptimos

Aplicamos varios ciclos de cuidado dentro de los rangos óptimos:
- Riego: 150 ml (sustrato mixto → 30 pts de agua)
- Luz: 50% (rango óptimo 30–70%)
- Ventilación: 55% (rango óptimo 40–75%)

**Esperamos:** salud → alta, etapa avanza.

In [6]:
def ciclo_optimo(n: int = 5):
    """Aplica N ciclos de cuidado óptimo y muestra la evolución."""
    print(f"\n{'─'*55}")
    print(f"  CICLOS ÓPTIMOS ({n} iteraciones)")
    print(f"{'─'*55}")
    print(f"{'Ciclo':>6} {'Agua':>6} {'Salud':>6} {'Etapa':<22} Alertas")
    print(f"{'─'*55}")

    for i in range(1, n + 1):
        requests.post(f"{BASE}/plants/{PLANT_ID}/water",
                      json={"cantidad_ml": 150}, headers=HEADERS)
        requests.post(f"{BASE}/plants/{PLANT_ID}/light",
                      json={"intensidad": 50}, headers=HEADERS)
        requests.post(f"{BASE}/plants/{PLANT_ID}/ventilation",
                      json={"nivel": 55}, headers=HEADERS)

        e = requests.get(f"{BASE}/plants/{PLANT_ID}/status", headers=HEADERS).json()
        p = e["planta"]
        alertas = ", ".join(e.get("alertas", [])) or "—"
        print(f"{i:>6} {p['water_level']:>6.1f} {p['health']:>6.1f} {p['growth_stage']:<22} {alertas}")

ciclo_optimo(6)


───────────────────────────────────────────────────────
  CICLOS ÓPTIMOS (6 iteraciones)
───────────────────────────────────────────────────────
 Ciclo   Agua  Salud Etapa                  Alertas
───────────────────────────────────────────────────────
     1   60.0  100.0 enraizamiento          —
     2   90.0   70.8 enraizamiento          💧 Agua elevada (90.0%): Pudrición de raíces, crecimiento de hongos en sustrato.
     3  100.0   65.0 plantula               ⚠️ AGUA EXCESIVA: Pudrición de raíces, crecimiento de hongos en sustrato.
     4  100.0   48.3 enraizamiento          ⚠️ AGUA EXCESIVA: Pudrición de raíces, crecimiento de hongos en sustrato., 🌫️ Humedad elevada (84.0%): Pudrición fúngica en raíces y sustrato.
     5  100.0   42.1 plantula               ⚠️ AGUA EXCESIVA: Pudrición de raíces, crecimiento de hongos en sustrato., 🌫️ Humedad elevada (90.0%): Pudrición fúngica en raíces y sustrato.
     6  100.0   40.0 plantula               ⚠️ AGUA EXCESIVA: Pudrición de raíces, c

---
## 5. 🔴 Pipeline de descuido — Riego excesivo (pudrición de raíces)

Regamos con cantidades exageradas repetidamente:
- Cada riego de 600 ml con sustrato mixto → +120 pts de agua  
- El agua sube por encima del límite óptimo (80%) y crítico (95%)
- La salud baja drásticamente (pudrición radicular)
- Con `health < 50` → crecimiento se **frena**
- Con `health < 30` → crecimiento **retrocede**

In [7]:
def ciclo_riego_excesivo(n: int = 8):
    """Aplica N riegos excesivos y muestra la degradación."""
    print(f"\n{'─'*70}")
    print(f"  RIEGO EXCESIVO ({n} riegos de 600 ml)")
    print(f"{'─'*70}")
    print(f"{'Riego':>6} {'Agua':>6} {'Humedad':>8} {'Salud':>6} {'Etiqueta':<12} {'Etapa':<24} Alertas")
    print(f"{'─'*70}")

    for i in range(1, n + 1):
        requests.post(f"{BASE}/plants/{PLANT_ID}/water",
                      json={"cantidad_ml": 600}, headers=HEADERS)

        e = requests.get(f"{BASE}/plants/{PLANT_ID}/status", headers=HEADERS).json()
        p = e["planta"]
        alertas = " | ".join(e.get("alertas", [])) or "—"
        print(
            f"{i:>6} {p['water_level']:>6.1f} {p['humidity_level']:>8.1f} "
            f"{p['health']:>6.1f} {e['salud_etiqueta']:<12} {p['growth_stage']:<24} {alertas}"
        )

ciclo_riego_excesivo(8)


──────────────────────────────────────────────────────────────────────
  RIEGO EXCESIVO (8 riegos de 600 ml)
──────────────────────────────────────────────────────────────────────
 Riego   Agua  Humedad  Salud Etiqueta     Etapa                    Alertas
──────────────────────────────────────────────────────────────────────
     1  100.0    100.0   40.0 Regular      plantula                 ⚠️ AGUA EXCESIVA: Pudrición de raíces, crecimiento de hongos en sustrato. | ⚠️ HUMEDAD EXCESIVA: Pudrición fúngica en raíces y sustrato.
     2  100.0    100.0   40.0 Regular      plantula                 ⚠️ AGUA EXCESIVA: Pudrición de raíces, crecimiento de hongos en sustrato. | ⚠️ HUMEDAD EXCESIVA: Pudrición fúngica en raíces y sustrato.
     3  100.0    100.0   40.0 Regular      plantula                 ⚠️ AGUA EXCESIVA: Pudrición de raíces, crecimiento de hongos en sustrato. | ⚠️ HUMEDAD EXCESIVA: Pudrición fúngica en raíces y sustrato.
     4  100.0    100.0   40.0 Regular      plantula      

### Comprobación visual del retroceso de etapa

Aquí vemos cómo la misma cantidad de `total_care_actions` produce etapas distintas según la salud:

| Salud | Factor de penalización | Ejemplo: 20 acciones → acciones efectivas |
|---|---|---|
| ≥ 50 | 100 % | 20 → etapa normal |
| 30–49 | 60 % | 12 → retrocede |
| < 30 | 25 % | 5 → retrocede severamente |

In [8]:
import sys, os
# Agregar el backend al path para importar directamente
BACKEND_PATH = os.path.join(os.path.dirname(os.getcwd()), "backend")
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

from plant.growth import obtener_etapa, calcular_salud, etiqueta_salud

print("Simulación local de penalización de crecimiento:")
print(f"{'Acciones':>9} {'Salud':>6} {'Etiqueta':<12} {'Efectivas':>9} {'Etapa':<22}")
print("─" * 65)

for total_acc in [5, 10, 15, 20, 30, 40, 50]:
    for health in [90.0, 45.0, 20.0]:
        etapa = obtener_etapa(total_acc, "orquidea", health)
        if health < 30:
            ef = max(0, int(total_acc * 0.25))
        elif health < 50:
            ef = max(0, int(total_acc * 0.60))
        else:
            ef = total_acc
        print(f"{total_acc:>9} {health:>6.0f} {etiqueta_salud(health):<12} {ef:>9} {etapa:<22}")
    print()

Simulación local de penalización de crecimiento:
 Acciones  Salud Etiqueta     Efectivas Etapa                 
─────────────────────────────────────────────────────────────────
        5     90 Excelente            5 enraizamiento         
        5     45 Regular              3 enraizamiento         
        5     20 Muy baja             1 germinacion           

       10     90 Excelente           10 plantula              
       10     45 Regular              6 enraizamiento         
       10     20 Muy baja             2 germinacion           

       15     90 Excelente           15 crecimiento           
       15     45 Regular              9 plantula              
       15     20 Muy baja             3 enraizamiento         

       20     90 Excelente           20 crecimiento           
       20     45 Regular             12 plantula              
       20     20 Muy baja             5 enraizamiento         

       30     90 Excelente           30 vara_floral           

---
## 6. 🔄 Recuperación — Dejar secar y restaurar condiciones

Reducimos el riego (solo 50 ml para no agregar más agua) y ajustamos a condiciones óptimas.
La ventilación alta ayuda a secar el exceso de agua del sustrato.

In [9]:
def ciclo_recuperacion(n: int = 8):
    """Aplica N ciclos de recuperación: poca agua, luz normal, ventilación alta para secar."""
    print(f"\n{'─'*65}")
    print(f"  RECUPERACIÓN ({n} ciclos: riego mínimo + ventilación alta)")
    print(f"{'─'*65}")
    print(f"{'Ciclo':>6} {'Agua':>6} {'Salud':>6} {'Etiqueta':<12} {'Etapa':<24} Alertas")
    print(f"{'─'*65}")

    for i in range(1, n + 1):
        # Riego mínimo (10 ml) para no agravar el encharcamiento
        requests.post(f"{BASE}/plants/{PLANT_ID}/water",
                      json={"cantidad_ml": 10}, headers=HEADERS)
        # Luz óptima
        requests.post(f"{BASE}/plants/{PLANT_ID}/light",
                      json={"intensidad": 50}, headers=HEADERS)
        # Ventilación alta para secar el sustrato encharcado
        requests.post(f"{BASE}/plants/{PLANT_ID}/ventilation",
                      json={"nivel": 85}, headers=HEADERS)

        e = requests.get(f"{BASE}/plants/{PLANT_ID}/status", headers=HEADERS).json()
        p = e["planta"]
        alertas = " | ".join(e.get("alertas", [])) or "—"
        print(
            f"{i:>6} {p['water_level']:>6.1f} {p['health']:>6.1f} "
            f"{e['salud_etiqueta']:<12} {p['growth_stage']:<24} {alertas}"
        )

ciclo_recuperacion(8)


─────────────────────────────────────────────────────────────────
  RECUPERACIÓN (8 ciclos: riego mínimo + ventilación alta)
─────────────────────────────────────────────────────────────────
 Ciclo   Agua  Salud Etiqueta     Etapa                    Alertas
─────────────────────────────────────────────────────────────────
     1   98.0   28.8 Muy baja     enraizamiento            ⚠️ AGUA EXCESIVA: Pudrición de raíces, crecimiento de hongos en sustrato. | ⚠️ HUMEDAD EXCESIVA: Pudrición fúngica en raíces y sustrato. | 💨 Ventilación elevada (85.0%): Sustrato se seca demasiado rápido, raíces deshidratadas.
     2   98.0   28.8 Muy baja     plantula                 ⚠️ AGUA EXCESIVA: Pudrición de raíces, crecimiento de hongos en sustrato. | ⚠️ HUMEDAD EXCESIVA: Pudrición fúngica en raíces y sustrato. | 💨 Ventilación elevada (85.0%): Sustrato se seca demasiado rápido, raíces deshidratadas.
     3   98.0   28.8 Muy baja     plantula                 ⚠️ AGUA EXCESIVA: Pudrición de raíces, creci

---
## 7. 🪴 Gestión de sustrato

El sustrato afecta cuánta agua absorbe la planta por cada ml regado.

| Sustrato | Factor retención | Efecto |
|---|---|---|
| `perlita` | 0.5× | Drena rápido, difícil encharcar |
| `corteza` | 0.7× | Equilibrado para orquídeas |
| `mixto` | 1.0× | Retención normal |
| `musgo_sphagnum` | 1.5× | Retiene mucho — riesgo de encharcamiento |

In [10]:
# Ver catálogo
sustratos = requests.get(f"{BASE}/plants/substrates", headers=HEADERS).json()
mostrar(sustratos, "Catálogo de sustratos")

# Cambiar a corteza (mejor para orquídeas — drena bien)
id_corteza = next(
    (s["id_substrate_type"] for s in sustratos.get("sustratos", [])
     if s["name"] == "corteza"), None
)

if id_corteza:
    resp = requests.put(
        f"{BASE}/plants/{PLANT_ID}/substrate",
        json={"substrate_type_id": id_corteza},
        headers=HEADERS,
    ).json()
    mostrar(resp, "Cambio de sustrato → corteza")


═══════════════════════════════════════════════════════
  Catálogo de sustratos
═══════════════════════════════════════════════════════
{
  "sustratos": [
    {
      "id_substrate_type": 1,
      "name": "corteza",
      "description": "Corteza de pino. Excelente drenaje, poca retención. Ideal para orquídeas.",
      "water_retention": 0.7,
      "nutrient_release": 0.8,
      "drainage_factor": 1.4
    },
    {
      "id_substrate_type": 2,
      "name": "musgo_sphagnum",
      "description": "Musgo sphagnum. Alta retención de humedad y agua.",
      "water_retention": 1.5,
      "nutrient_release": 1.2,
      "drainage_factor": 0.6
    },
    {
      "id_substrate_type": 3,
      "name": "perlita",
      "description": "Perlita volcánica. Drenaje máximo, retención mínima.",
      "water_retention": 0.5,
      "nutrient_release": 0.5,
      "drainage_factor": 1.8
    },
    {
      "id_substrate_type": 4,
      "name": "mixto",
      "description": "Mezcla balanceada para uso genera

## 8. 🏺 Maceta

La maceta tiene propiedades físicas independientes.
- `material`: plástico, cerámica, terracota, vidrio
- `drainage_level`: cuánto drena la maceta (campo registrado)
- `ventilation_level` de la maceta: huecos físicos (diferente al nivel ambiental)

In [11]:
# Ver maceta actual
maceta = requests.get(f"{BASE}/plants/{PLANT_ID}/pot", headers=HEADERS).json()
mostrar(maceta, "Maceta actual")

# Cambiar a terracota — mejor para orquídeas (permite respirar las raíces)
resp = requests.put(
    f"{BASE}/plants/{PLANT_ID}/pot",
    json={"material": "terracota", "drainage_level": 75.0},
    headers=HEADERS,
).json()
mostrar(resp, "Maceta → terracota (mejor drenaje)")


═══════════════════════════════════════════════════════
  Maceta actual
═══════════════════════════════════════════════════════
{
  "maceta": {
    "id_pot": 1,
    "material": "plastico",
    "size_cm": 15,
    "drainage_level": 60.0,
    "ventilation_level": 50.0,
    "fk_plant_id": 1
  }
}

═══════════════════════════════════════════════════════
  Maceta → terracota (mejor drenaje)
═══════════════════════════════════════════════════════
{
  "mensaje": "Maceta actualizada correctamente.",
  "maceta": {
    "id_pot": 1,
    "material": "terracota",
    "size_cm": 15,
    "drainage_level": 75.0,
    "ventilation_level": 50.0
  }
}


---
## 9. 📋 Historial de cuidados

Cada acción queda registrada automáticamente: tipo, valor, notas y timestamp.

In [12]:
historial = requests.get(
    f"{BASE}/plants/{PLANT_ID}/history",
    params={"limit": 10},
    headers=HEADERS,
).json()

print(f"Total de acciones registradas: {historial.get('total', 0)}")
print()
print(f"{'Tipo':<14} {'Valor':>8}  {'Nota':<50} Timestamp")
print("─" * 90)
for h in historial.get("historial", []):
    print(f"{h['action_type']:<14} {h['value']:>8.1f}  {str(h['extra_info'] or ''):<50} {h['created_at']}")

Total de acciones registradas: 52

Tipo              Valor  Nota                                               Timestamp
──────────────────────────────────────────────────────────────────────────────────────────
maceta              0.0  Maceta actualizada: material=terracota, drainage_level=75.0 2026-04-19 23:28:52
sustrato            1.0  Sustrato cambiado a: corteza                       2026-04-19 23:28:40
luz                50.0  Intensidad: 50%                                    2026-04-19 23:28:27
ventilacion        85.0  Ventilación ajustada a 85%                         2026-04-19 23:28:27
riego              10.0  +2.0 pts agua (sustrato: mixto, retención: 1.00x)  2026-04-19 23:28:27
ventilacion        85.0  Ventilación ajustada a 85%                         2026-04-19 23:28:26
riego              10.0  +2.0 pts agua (sustrato: mixto, retención: 1.00x)  2026-04-19 23:28:26
luz                50.0  Intensidad: 50%                                    2026-04-19 23:28:26
luz        

---
## 10. 📊 Resumen de reglas de penalización

Esta celda muestra todos los umbrales críticos del simulador de forma consolidada.

In [13]:
from plant.growth import (
    REGLAS_AGUA, REGLAS_LUZ, REGLAS_HUMEDAD, REGLAS_VENTILACION,
    calcular_salud, etiqueta_salud
)

print("═" * 70)
print("  TABLA DE UMBRALES Y EFECTOS")
print("═" * 70)
for nombre, reglas in [
    ("AGUA",        REGLAS_AGUA),
    ("LUZ",         REGLAS_LUZ),
    ("HUMEDAD",     REGLAS_HUMEDAD),
    ("VENTILACIÓN", REGLAS_VENTILACION),
]:
    print(f"""
{nombre}:
  Óptimo : {reglas['optimo_bajo']}% – {reglas['optimo_alto']}%   (salud = 100 pts)
  Crítico: <{reglas['critico_bajo']}% o >{reglas['critico_alto']}%  (salud = 0 pts)
  Efecto bajo   : {reglas['efecto_bajo']}
  Efecto exceso : {reglas['efecto_alto']}"""
)

print()
print("═" * 70)
print("  PENALIZACIÓN DE CRECIMIENTO POR SALUD")
print("═" * 70)
print("  salud ≥ 50  → crecimiento normal       (100% de acciones efectivas)")
print("  30 ≤ salud < 50 → crecimiento lento    ( 60% de acciones efectivas)")
print("  salud < 30  → crecimiento revertido     ( 25% de acciones efectivas)")

print()
print("═" * 70)
print("  FÓRMULA DE SALUD")
print("═" * 70)
print("  health = agua×35% + luz×25% + humedad×25% + ventilación×15%")
print()
print("  Ejemplos:")
casos = [
    (60, 50, 70, 55, "Condiciones óptimas"),
    (95, 50, 85, 55, "Agua excesiva (encharcamiento)"),
    (15, 50, 65, 55, "Agua insuficiente (sequía)"),
    (60, 95, 40, 55, "Luz excesiva"),
    (60, 50, 70, 10, "Sin ventilación"),
]
print(f"  {'Caso':<32} {'Agua':>5} {'Luz':>4} {'Hum':>4} {'Vent':>5} {'Salud':>6} Etiqueta")
print("  " + "─" * 65)
for w, l, h, v, desc in casos:
    s = calcular_salud(w, l, h, v)
    print(f"  {desc:<32} {w:>5} {l:>4} {h:>4} {v:>5} {s:>6.1f} {etiqueta_salud(s)}")

══════════════════════════════════════════════════════════════════════
  TABLA DE UMBRALES Y EFECTOS
══════════════════════════════════════════════════════════════════════

AGUA:
  Óptimo : 40.0% – 80.0%   (salud = 100 pts)
  Crítico: <10.0% o >95.0%  (salud = 0 pts)
  Efecto bajo   : Marchitamiento de hojas, raíces en estrés hídrico.
  Efecto exceso : Pudrición de raíces, crecimiento de hongos en sustrato.

LUZ:
  Óptimo : 30.0% – 70.0%   (salud = 100 pts)
  Crítico: <5.0% o >90.0%  (salud = 0 pts)
  Efecto bajo   : Etiolación del tallo, hojas pálidas y débiles.
  Efecto exceso : Quemaduras en hojas, desecación acelerada del sustrato.

HUMEDAD:
  Óptimo : 50.0% – 80.0%   (salud = 100 pts)
  Crítico: <20.0% o >92.0%  (salud = 0 pts)
  Efecto bajo   : Raíces deshidratadas, hojas enrolladas.
  Efecto exceso : Pudrición fúngica en raíces y sustrato.

VENTILACIÓN:
  Óptimo : 40.0% – 75.0%   (salud = 100 pts)
  Crítico: <10.0% o >95.0%  (salud = 0 pts)
  Efecto bajo   : Raíces sin oxígeno, 